In [2]:
import math
import cmath
from functools import lru_cache

# Optional: NumPy for scaling fit
try:
    import numpy as np
except ImportError:
    np = None

# Optional: SymPy for classical 6j reference
try:
    import sympy as sp
    from sympy.physics.wigner import wigner_6j
    HAVE_SYMPY = True
except Exception:
    HAVE_SYMPY = False

EPS_THETA = 1e-10


# ==========================
# q-number and q-factorial
# ==========================

def q_number(n: int, theta: float) -> complex:
    """
    [n]_q = sin(n*theta) / sin(theta), q = e^{i theta}
    For small theta, returns n.
    """
    if n < 0:
        raise ValueError("q_number: n must be >= 0")

    # theta ~ 0 or 2π -> classical limit
    if abs(theta) < EPS_THETA or abs(abs(theta) - 2 * math.pi) < EPS_THETA:
        return complex(n)

    # theta ~ pi (q ≈ -1), integer n has removable singularity
    if abs(abs(theta) - math.pi) < EPS_THETA:
        # limit of sin(nθ)/sin θ as θ→π is n*(-1)^(n+1)
        return complex(n * ((-1) ** (n + 1)))

    den = math.sin(theta)
    if abs(den) < 1e-16:
        return complex(n)

    num = math.sin(n * theta)
    return complex(num / den)


@lru_cache(maxsize=None)
def log_q_factorial(n: int, theta: float) -> complex:
    """
    log([n]_q!) as complex a+ib, where [n]_q! = ∏_{k=1}^n [k]_q.
    For theta ≈ 0, uses real lgamma for speed.
    Returns -inf+0j if the product effectively vanishes.
    """
    if n < 0:
        return complex(float("-inf"), 0.0)
    if n == 0:
        return complex(0.0, 0.0)

    # classical limit
    if abs(theta) < EPS_THETA or abs(abs(theta) - 2 * math.pi) < EPS_THETA:
        return complex(math.lgamma(n + 1.0), 0.0)

    log_mag = 0.0
    phase = 0.0
    for k in range(1, n + 1):
        z = q_number(k, theta)
        r = abs(z)
        if r < 1e-30:
            return complex(float("-inf"), 0.0)
        log_mag += math.log(r)
        phase += cmath.phase(z)
    return complex(log_mag, phase)


def safe_exp_log(z: complex) -> complex:
    """
    For z = a+ib, return exp(z).
    If a=-inf, return 0.
    """
    if math.isinf(z.real) and z.real < 0:
        return 0j
    return cmath.rect(math.exp(z.real), z.imag)


# ==========================
# q-triangle coefficient
# ==========================

@lru_cache(maxsize=None)
def log_q_delta(a: float, b: float, c: float, theta: float) -> complex:
    """
    log(Δ_q(a,b,c)), where
    Δ_q(a,b,c) = sqrt( [a+b-c]! [a-b+c]! [-a+b+c]! / [a+b+c+1]! ).
    Enforces SU(2) triangle + parity rules.
    """
    # non-negativity
    if a < 0 or b < 0 or c < 0:
        return complex(float("-inf"), 0.0)

    # triangle inequalities
    if (a + b < c) or (a + c < b) or (b + c < a):
        return complex(float("-inf"), 0.0)

    # parity: a+b+c must be integer
    if abs((a + b + c) - round(a + b + c)) > 1e-8:
        return complex(float("-inf"), 0.0)

    # factorial arguments (integers)
    n1 = int(round(a + b - c))
    n2 = int(round(a - b + c))
    n3 = int(round(-a + b + c))
    n4 = int(round(a + b + c + 1))

    if min(n1, n2, n3) < 0:
        return complex(float("-inf"), 0.0)

    lm1 = log_q_factorial(n1, theta)
    lm2 = log_q_factorial(n2, theta)
    lm3 = log_q_factorial(n3, theta)
    lm4 = log_q_factorial(n4, theta)

    if any(math.isinf(z.real) and z.real < 0 for z in (lm1, lm2, lm3, lm4)):
        return complex(float("-inf"), 0.0)

    return 0.5 * (lm1 + lm2 + lm3 - lm4)


# ==========================
# q-deformed 6j via Racah
# ==========================

def sixj_q(j1, j2, j3, j4, j5, j6, theta: float) -> complex:
    """
    q-deformed SU(2) 6j symbol with q = e^{i theta}.
    j's are half-integers (floats or ints).
    """
    # 1. prefactor from four triangles
    deltas = [
        log_q_delta(j1, j2, j3, theta),
        log_q_delta(j1, j5, j6, theta),
        log_q_delta(j4, j2, j6, theta),
        log_q_delta(j4, j5, j3, theta),
    ]
    if any(math.isinf(d.real) and d.real < 0 for d in deltas):
        return 0j
    log_pref = sum(deltas)

    # 2. Racah t-range
    t_min = max(
        j1 + j2 + j3,
        j1 + j5 + j6,
        j4 + j2 + j6,
        j4 + j5 + j3,
    )
    t_max = min(
        j1 + j2 + j4 + j5,
        j1 + j3 + j4 + j6,
        j2 + j3 + j5 + j6,
    )
    t_start = int(math.ceil(t_min - 1e-8))
    t_end = int(math.floor(t_max + 1e-8))
    if t_start > t_end:
        return 0j

    sum_val = 0j

    for t in range(t_start, t_end + 1):
        # numerator [t+1]_q!
        ln_num = log_q_factorial(t + 1, theta)
        if math.isinf(ln_num.real) and ln_num.real < 0:
            continue

        # denominator: 7 q-factorials
        args
